In [ ]:
import os
import re
import pandas as pd
from pathlib import Path

data_dir = Path(r"C:\Users\New\Documents\DTS201-GroupJ\data\raw_2025_data")
output_file = Path(r"C:\Users\New\Documents\DTS201-GroupJ\data\cleaned_foodprice_2024_2026.csv")
all_dfs = []

# List of food items
selected_items = [
    "Rice Local, Short-Grained",
    "Rice Long-Grained (Imported)",
    "Garri Yellow",
    "Beans White",
    "Yam Tuber",
    "Palm Oil, 75Cl",
    "Tomatoes, Fresh",
    "Beef Boneless",
    "Onions, Fresh",
    "Agric Hen Eggs",
    "Bread (Sliced), 450G",
    "Sweet Potatoes",
    "Watermelon Whole, Medium Size Fresh",
    "Vegetable Oil, 75Cl"
]

month_map = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12
}

# Loop through all .xlsx files in the folder
for file in sorted(os.listdir(data_dir)):
    if file.endswith(".xlsx") and not file.startswith("~$"):
        file_path = os.path.join(data_dir, file)
        
        # Extract Month and Year from filename
        match = re.match(r"([A-Za-z]+)(\d{4})\.xlsx", file)
        if not match:
            print(f"Skipping file with unexpected name format: {file}")
            continue
            
        month_name, year_str = match.groups()
        month_title = month_name.title()
        
        # Read the first sheet of each Excel workbook
        df = pd.read_excel(file_path, sheet_name=0)
        df.columns = df.columns.str.strip()
        
        # Standardize Item Labels
        if 'Item Labels' in df.columns:
            df = df.rename(columns={'Item Labels': 'Item Label'})
            
        # Standardize the current month's average price column
        current_price_col = df.columns[3]
        df = df.rename(columns={current_price_col: 'Avg_Price'})
        
        # Keep only the clean, consistent columns so pd.concat never crashes
        keep_cols = [c for c in ['Item Label', 'Avg_Price', 'MoM', 'YoY', 'Highest', 'Lowest'] if c in df.columns]
        df = df[keep_cols].copy()
        
        df['Month'] = month_title
        df['Month_Num'] = month_map.get(month_title, 0)
        df['Year'] = int(year_str)
        
        all_dfs.append(df)

# Combine all data
if all_dfs:
    master_df = pd.concat(all_dfs, ignore_index=True)
    
    # Remove whitespace and trailing commas
    master_df['Item Label'] = (
        master_df['Item Label']
        .astype(str)
        .str.strip()
        .str.rstrip(',')
        .str.strip()
        .str.title()
    )
    
    # Drop rows without item label
    master_df = master_df.dropna(subset=['Item Label'])
    
    # Select specific food items
    master_df = master_df[master_df['Item Label'].isin(selected_items)].copy()
    
    # Sort chronologically using Year and Month_Num (no Date column)
    master_df = master_df.sort_values(['Item Label', 'Year', 'Month_Num']).reset_index(drop=True)
    
    # Save cleaned combined data
    master_df.to_csv(output_file, index=False)
    print(f"Success! Saved {len(master_df)} rows to {output_file}")
else:
    print("No files found. Check your folder path!")


Success! Saved 238 rows to C:\Users\New\Documents\DTS201-GroupJ\data\cleaned_foodprice_2024_2026.csv
